# **DATA PREPROCESSING**

# **Handle missing values (remove or fill)**

In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('Kolesa_Raw_Data.csv')

1. **Drop columns with excessive missing data** (Requirement: Remove). These columns are missing *>91%* of data and are not relevant for market price analysis

In [ ]:
cols_to_drop = ['monthly_payment', 'down_payment_pct', 'down_payment']
df.drop(columns=cols_to_drop, inplace=True)

2. **Clean and fill mileage** (Requirement: Fill by Constant). Convert string to integer and fill NaNs with 0 (assuming brand new for luxury segment)

In [ ]:
df['mileage'] = df['mileage'].fillna('0').str.replace(r'\D', '', regex=True).astype(int)

3. **Handle Mean Price for bargain analysis** (Requirement: Fill by Median). Remove currency symbols/spaces and fill missing values with the column median

In [ ]:
df['mean_price'] = df['mean_price'].str.replace(r'\D', '', regex=True).astype(float)
df['mean_price'] = df['mean_price'].fillna(df['mean_price'].median())

4. **Handle categorical missing values**. Label missing colors and engine details to avoid NULLs in SQL

In [ ]:
df['color'] = df['color'].fillna('Не указан')
df['engine_volume'] = df['engine_volume'].fillna('Не указан')

Verify that key columns no longer contain null values

In [ ]:
print("Анализ после обработки пропусков:")
print(df[['year', 'mileage', 'mean_price']].isnull().sum())

Анализ после обработки пропусков:
year          0
mileage       0
mean_price    0
dtype: int64


# **Remove Duplicates**

Record the number of rows before cleaning

In [ ]:
initial_count = len(df)

**Step A**: Remove exact duplicates based on the unique URL (technical errors)

In [ ]:
df.drop_duplicates(subset=['link'], keep='first', inplace=True)

**Step B:** Remove content duplicates (Same car posted multiple times: same title, year, price, and city)

In [ ]:
df.drop_duplicates(subset=['title', 'year', 'price', 'city'], keep='first', inplace=True)

Calculate how many duplicates were removed

In [ ]:
removed_count = initial_count - len(df)
print(f"Duplicates removed: {removed_count}")
print(f"Current rows in dataset: {len(df)}")

Duplicates removed: 181
Current rows in dataset: 1019


# **Fix data types**

**A. Convert Price to numeric.** Using regex \D to remove all non-numeric characters (currency symbols, spaces)

In [ ]:
df['price'] = df['price'].astype(str).str.replace(r'\D', '', regex=True)
df['price'] = pd.to_numeric(df['price'], errors='coerce').astype(int)

**B. Convert Mileage to numeric.** Already handled NaNs in the previous step, now stripping units and converting

In [ ]:
df['mileage'] = df['mileage'].astype(str).str.replace(r'\D', '', regex=True)
df['mileage'] = pd.to_numeric(df['mileage'], errors='coerce').astype(int)

**C. Extract numeric Engine Volume.** Extracting float values (e.g., 3.5 from "3.5 (petrol)"). Fill missing engine volumes with the median for the luxury segment

In [ ]:
df['engine_volume_num'] = df['engine_volume'].astype(str).str.extract(r'(\d+\.?\d*)').astype(float)
df['engine_volume_num'] = df['engine_volume_num'].fillna(df['engine_volume_num'].median())

In [ ]:
print("Data types conversion complete:")
print(df[['price', 'mileage', 'engine_volume_num']].dtypes)

Data types conversion complete:
price                  int64
mileage                int64
engine_volume_num    float64
dtype: object


# **Clean Text**

List of columns that need text normalization

In [ ]:
text_columns = ['title', 'city', 'color', 'bodywork', 'transmission', 'drive', 'wheel']

In [ ]:
for col in text_columns:
    # A. Remove leading/trailing whitespaces and hidden characters
    df[col] = df[col].astype(str).str.strip()

    # B. Normalize formats to Title Case (e.g., 'ALMATY' -> 'Almaty')
    # This ensures consistency during SQL GROUP BY operations
    df[col] = df[col].str.capitalize()

# C. Specific cleaning for 'title'
# Removing any redundant technical symbols that might interfere with readability
df['title'] = df['title'].str.replace(r'[^\w\s\d-]', '', regex=True)

print("Text data normalized and cleaned.")
print(df[['title', 'city', 'color']].head())

Text data normalized and cleaned.
                    title      city           color
0               Dodge ram  Кокшетау  Белый металлик
1  Lexus lx 600 overtrail    Астана           Белый
2                  Bmw x5  Павлодар       Не указан
3       Bmw x5 xdrive 40i  Павлодар       Не указан
4            Lexus lx 600    Астана       Не указан


# **Create new features**

**A. Extract Brand from Title**. Taking the first word of the title (e.g., "Lexus LX 600" -> "Lexus")

In [ ]:
df['brand'] = df['title'].str.split().str[0]

**B. Calculate Price Difference** *(Market Value vs Seller Price)*. This shows how much "profit" or "discount" a buyer gets compared to the average

In [ ]:
df['price_diff'] = df['mean_price'] - df['price']

**C. Create a "Bargain" Category**. If the price is 5% lower than the market average, we mark it as a bargain

In [ ]:
df['is_bargain'] = np.where(df['price'] < (df['mean_price'] * 0.95), 'Yes', 'No')

**D. Calculate Car Age**. Using the recovered 'year' to find out how old the vehicle is

In [ ]:
current_year = 2026
df['car_age'] = current_year - df['year']

**E. Price per Age Year (Price efficiency)**. Avoid division by zero for brand new cars

In [ ]:
df['price_per_year'] = df['price'] / (df['car_age'] + 1)

In [ ]:
print("Feature engineering complete. New columns added: brand, price_diff, is_bargain, car_age.")
print(df[['title', 'brand', 'price_diff', 'is_bargain', 'car_age']].head())

Feature engineering complete. New columns added: brand, price_diff, is_bargain, car_age.
                    title  brand  price_diff is_bargain  car_age
0               Dodge ram  Dodge -11776000.0         No        8
1  Lexus lx 600 overtrail  Lexus   8802000.0        Yes        5
2                  Bmw x5    Bmw -12427000.0         No        3
3       Bmw x5 xdrive 40i    Bmw  -6672000.0         No        3
4            Lexus lx 600  Lexus   -945000.0         No        5


# **DATASET COMPARISON**

In [ ]:
print(f"Rows before: {initial_count}")
print(f"Rows after:  {len(df)}")
print(f"Columns removed: ['monthly_payment', 'down_payment_pct', 'down_payment']")
print(f"New columns created: ['brand', 'price_diff', 'is_bargain', 'car_age']")

comparison_df = pd.DataFrame({
    'Column': ['price', 'mileage', 'year'],
    'Before (Type)': ['object (string)', 'object (string)', 'None/Missing'],
    'After (Type)': [df['price'].dtype, df['mileage'].dtype, df['year'].dtype]
})
print("\nType Transformation:")
print(comparison_df)

Rows before: 1200
Rows after:  1019
Columns removed: ['monthly_payment', 'down_payment_pct', 'down_payment']
New columns created: ['brand', 'price_diff', 'is_bargain', 'car_age']

Type Transformation:
    Column    Before (Type) After (Type)
0    price  object (string)        int64
1  mileage  object (string)        int64
2     year     None/Missing        int64


In [ ]:
df.to_csv('Kolesa_Final_Cleaned.csv', index=False, encoding='utf-8-sig')